# TP2 - Reconstruccion 3D

In [ ]:
import glob
import os
import pickle
import sys
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import open3d as o3d

sys.path.append("code_examples/stereodemo_disp_example")
from disparity.method_cre_stereo import CREStereo
from disparity.methods import Config, InputPair

np.set_printoptions(precision=4, suppress=True)

DATA_DIR = "i308-calib/data"
CALIB_DIR = os.path.join(DATA_DIR, "calib")
CAPTURES_DIR = os.path.join(DATA_DIR, "captures")
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

# checkerboard usado en la calibracion
CHECKERBOARD = (9, 6)
SQUARE_SIZE = 28.0

## 4.1 Calibracion y rectificacion estereo

### Parametros Obtenidos (Calibracion hecha en tutorial)

In [ ]:
with open(os.path.join(DATA_DIR, "stereo_calibration.pkl"), "rb") as f:
    calib = pickle.load(f)
with open(os.path.join(DATA_DIR, "stereo_maps.pkl"), "rb") as f:
    maps = pickle.load(f)

K_l, dist_l = calib["left_K"], calib["left_dist"]
K_r, dist_r = calib["right_K"], calib["right_dist"]
R, T = calib["R"], calib["T"]
P1, P2, Q = maps["P1"], maps["P2"], maps["Q"]

for name in ["left_K", "left_dist", "right_K", "right_dist", "R", "T"]:
    print(f"{name}:\n{calib[name]}\n")
for name in ["R1", "R2", "P1", "P2", "Q"]:
    print(f"{name}:\n{maps[name]}\n")

print("image size:", calib["image_size"])
print(f"baseline: {np.linalg.norm(T):.2f} mm")
print(f"rotación entre cámaras: {np.degrees(np.linalg.norm(cv2.Rodrigues(R)[0])):.3f}°")

### Error de reproyeccion

In [ ]:
def numeric_key(path):
    return int(path.rsplit("_", 1)[-1].split(".")[0])

# calcula la pos en px de las esquinas del checkerboard
def detect_checkerboard(gray):
    flags = cv2.CALIB_CB_ADAPTIVE_THRESH | cv2.CALIB_CB_NORMALIZE_IMAGE | cv2.CALIB_CB_FAST_CHECK
    _, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, flags=flags)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
    corners = cv2.cornerSubPix(gray, corners, (7, 7), (-1, -1), criteria)
    return corners.reshape(-1, 2)

obj_points = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
obj_points[:, :2] = SQUARE_SIZE * np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)

pair_ids, corners_l, corners_r = [], [], []
for left_path in sorted(glob.glob(os.path.join(CALIB_DIR, "left_*.jpg")), key=numeric_key):
    pair_ids.append(numeric_key(left_path))
    corners_l.append(detect_checkerboard(cv2.imread(left_path, cv2.IMREAD_GRAYSCALE)))
    corners_r.append(detect_checkerboard(cv2.imread(left_path.replace("left_", "right_"), cv2.IMREAD_GRAYSCALE)))

# intrinsecos del pkl fijos, solo se reajusta la pose relativa, que queda igual a la guardada
rms_stereo, *_, R_check, T_check, _, _, _, _, per_view_errors = cv2.stereoCalibrateExtended(
    [obj_points] * len(pair_ids), corners_l, corners_r, K_l, dist_l, K_r, dist_r, calib["image_size"],
    R.copy(), T.copy(), flags=cv2.CALIB_FIX_INTRINSIC
)

print(f"pares usados: {len(pair_ids)}")
print(f"RMS estéreo: {rms_stereo:.3f} px")
print(f"diferencia con el pkl: R {np.abs(R_check - R).max():.5f}, T {np.abs(T_check - T).max():.3f} mm")

In [ ]:
x = np.arange(len(pair_ids))
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(x - 0.2, per_view_errors[:, 0], width=0.4, label="izquierda")
ax.bar(x + 0.2, per_view_errors[:, 1], width=0.4, label="derecha")
ax.axhline(rms_stereo, color="k", ls="--", lw=1, label=f"RMS estéreo = {rms_stereo:.3f} px")
ax.set_xticks(x)
ax.set_xticklabels(pair_ids)
ax.set_xlabel("par de calibración")
ax.set_ylabel("RMS [px]")
ax.set_title("Error de reproyección por par")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "reprojection_error.png"), dpi=150)
plt.show()

### Pares originales vs rectificados

In [ ]:
PAIR_INDEX = 0

img_l = cv2.imread(os.path.join(CAPTURES_DIR, f"left_{PAIR_INDEX}.jpg"))
img_r = cv2.imread(os.path.join(CAPTURES_DIR, f"right_{PAIR_INDEX}.jpg"))
rect_l = cv2.remap(img_l, maps["left_map_x"], maps["left_map_y"], cv2.INTER_LINEAR)
rect_r = cv2.remap(img_r, maps["right_map_x"], maps["right_map_y"], cv2.INTER_LINEAR)

images = [img_l, img_r, rect_l, rect_r]
titles = ["izquierda original", "derecha original", "izquierda rectificada", "derecha rectificada"]
line_ys = np.linspace(60, img_l.shape[0] - 60, 14)
colors = plt.cm.tab10(np.arange(len(line_ys)) % 10)

fig, axes = plt.subplots(2, 2, figsize=(16, 9.5))
for ax, img, title in zip(axes.ravel(), images, titles):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    for y, c in zip(line_ys, colors):
        ax.axhline(y, color=c, lw=0.8)
    ax.set_title(title)
    ax.axis("off")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, f"rectification_pair_{PAIR_INDEX}.png"), dpi=150)
plt.show()

## 4.2 Disparidad, profundidad y pose de camaras

### 4.2.1 Disparidad

In [ ]:
def load_rectified_pair(index):
    img_l = cv2.imread(os.path.join(CAPTURES_DIR, f"left_{index}.jpg"))
    img_r = cv2.imread(os.path.join(CAPTURES_DIR, f"right_{index}.jpg"))
    rect_l = cv2.remap(img_l, maps["left_map_x"], maps["left_map_y"], cv2.INTER_LINEAR)
    rect_r = cv2.remap(img_r, maps["right_map_x"], maps["right_map_y"], cv2.INTER_LINEAR)
    return rect_l, rect_r

# guardamos los pares rectificados
capture_ids = sorted(numeric_key(p) for p in glob.glob(os.path.join(CAPTURES_DIR, "left_*.jpg")))
rectified = {i: load_rectified_pair(i) for i in capture_ids}
print("pares rectificados:", len(rectified))

In [ ]:
SCALE = 0.5
NUM_DISPARITIES = 160

# downscaleamos la imagen a la mitad (para BM y SGBM)
def downscale(img):
    return cv2.resize(img, None, fx=SCALE, fy=SCALE, interpolation=cv2.INTER_AREA)

# upsacleamos la imagen
def to_full_resolution(disp_fixed_point, shape):
    disp = disp_fixed_point.astype(np.float32) / 16.0
    return cv2.resize(disp, (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST) / SCALE

def disparity_bm(rect_l, rect_r):
    bm = cv2.StereoBM_create(numDisparities=NUM_DISPARITIES, blockSize=11) # ventana de 11x11 px
    bm.setUniquenessRatio(10) # el mejor candidato tiene que ser al menos un 10% mejor que el segundo, sino se queda en negro
    
    bm.setSpeckleWindowSize(100) # eliminan ruido
    bm.setSpeckleRange(2)
    
    gray_l = cv2.cvtColor(downscale(rect_l), cv2.COLOR_BGR2GRAY)
    gray_r = cv2.cvtColor(downscale(rect_r), cv2.COLOR_BGR2GRAY)
    return to_full_resolution(bm.compute(gray_l, gray_r), rect_l.shape)

def disparity_sgbm(rect_l, rect_r):
    block_size = 5
    sgbm = cv2.StereoSGBM_create(
        minDisparity=0,
        numDisparities=NUM_DISPARITIES,
        blockSize=block_size,
        P1=8 * 3 * block_size**2, # penalizacion si un vecino difiere en un px de disparidad
        P2=32 * 3 * block_size**2, # penalizacion si difiere en mas de un px
        disp12MaxDiff=1,
        uniquenessRatio=10,
        speckleWindowSize=100,
        speckleRange=2,
        mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY,
    )
    return to_full_resolution(sgbm.compute(downscale(rect_l), downscale(rect_r)), rect_l.shape)

# red (jijo)
cre_stereo = CREStereo(Config(models_path=Path.home() / ".cache" / "stereodemo" / "models"))
cre_stereo.parameters["Mode"].set_value("combined")
cre_stereo.parameters["Shape"].set_value("1280x720")
cre_stereo.parameters["Iterations"].set_value("10")

def disparity_crestereo(rect_l, rect_r):
    return cre_stereo.compute_disparity(InputPair(rect_l, rect_r, None)).disparity_pixels

METHODS = {"bm": disparity_bm, "sgbm": disparity_sgbm, "crestereo": disparity_crestereo}

In [ ]:
disparities = {name: {} for name in METHODS}
times = {name: [] for name in METHODS}
for name, compute_disparity in METHODS.items():
    for i in capture_ids:
        start = time.time()
        disparities[name][i] = compute_disparity(*rectified[i])
        times[name].append(time.time() - start)

print(f"{'método':<10} {'% válidos':>10} {'tiempo/par [s]':>15}")
for name in METHODS:
    valid = 100 * np.mean([np.mean(d > 0) for d in disparities[name].values()])
    print(f"{name:<10} {valid:>10.1f} {np.mean(times[name]):>15.2f}")
print(f"disparidad máxima de CREStereo (percentil 99.9): {max(np.percentile(d, 99.9) for d in disparities['crestereo'].values()):.0f} px")

In [ ]:
SHOW_IDS = [0, 7]

cmap = plt.colormaps["turbo"].with_extremes(bad="black")
fig, axes = plt.subplots(len(SHOW_IDS), 1 + len(METHODS), figsize=(20, 3.3 * len(SHOW_IDS)), layout="constrained")
for row, i in zip(axes, SHOW_IDS):
    row[0].imshow(cv2.cvtColor(rectified[i][0], cv2.COLOR_BGR2RGB))
    row[0].set_title(f"par {i}")
    for ax, name in zip(row[1:], METHODS):
        disp = np.where(disparities[name][i] > 0, disparities[name][i], np.nan)
        im = ax.imshow(disp, cmap=cmap, vmin=0, vmax=NUM_DISPARITIES / SCALE)
        ax.set_title(name)
    for ax in row:
        ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.8, label="disparidad [px]")
fig.savefig(os.path.join(FIG_DIR, "disparity_methods.png"), dpi=150)
plt.show()

### 4.2.2 Nube de puntos local

In [ ]:
MAX_DEPTH = 1000.0
MIN_COLUMN = int(NUM_DISPARITIES / SCALE)
MAX_DISP_GRADIENT = 1.0

# calcula la nube de puntos a partir de disparidades
def local_point_cloud(disparity, rect_l, max_gradient=MAX_DISP_GRADIENT):
    points = cv2.reprojectImageTo3D(disparity, Q)
    gradient = np.hypot(cv2.Sobel(disparity, cv2.CV_32F, 1, 0), cv2.Sobel(disparity, cv2.CV_32F, 0, 1)) / 8
    valid = (disparity > 0) & (points[:, :, 2] < MAX_DEPTH) & (gradient < max_gradient) # no negros, puntos muy lejos ni bordes
    valid[:, :MIN_COLUMN] = False # puntos que caen fuera de la imagen derecha

    cloud = o3d.geometry.PointCloud()
    cloud.points = o3d.utility.Vector3dVector(points[valid].astype(np.float64))
    cloud.colors = o3d.utility.Vector3dVector(cv2.cvtColor(rect_l, cv2.COLOR_BGR2RGB)[valid] / 255.0)
    return cloud

# crea camara virtual que mira al centro desde "distance" mm girada por "yaw" y "pitch"
def view_extrinsic(center, yaw, pitch, distance):
    R = cv2.Rodrigues(np.radians([0, yaw, 0]))[0] @ cv2.Rodrigues(np.radians([pitch, 0, 0]))[0]
    extrinsic = np.eye(4)
    extrinsic[:3, :3] = R
    extrinsic[:3, 3] = [0, 0, distance] - R @ center
    return extrinsic

# renderiza en una ventana oculta de Open3D y devuelve la imagen
def render(geometries, extrinsic, width=1280, height=720, point_size=2.0):
    vis = o3d.visualization.Visualizer()
    vis.create_window(visible=False, width=width, height=height)
    for g in geometries:
        vis.add_geometry(g)
    vis.get_render_option().point_size = point_size
    view = vis.get_view_control()
    params = view.convert_to_pinhole_camera_parameters()
    params.extrinsic = extrinsic
    view.convert_from_pinhole_camera_parameters(params, allow_arbitrary=True)
    image = np.asarray(vis.capture_screen_float_buffer(do_render=True))
    vis.destroy_window()
    return np.clip(image, 0, 1)

In [ ]:
COMPARE_ID = 7

compare_clouds = {name: local_point_cloud(disparities[name][COMPARE_ID], rectified[COMPARE_ID][0]) for name in METHODS}
for name, cloud in compare_clouds.items():
    print(f"{name}: {len(cloud.points)} puntos")

camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=50)
center = np.median(np.asarray(compare_clouds["crestereo"].points), axis=0)
VIEWS = {"frontal": view_extrinsic(center, 0, 0, 450), "oblicua": view_extrinsic(center, 50, -25, 450)}

fig, axes = plt.subplots(len(METHODS), len(VIEWS), figsize=(16, 4.6 * len(METHODS)), layout="constrained")
for row, (name, cloud) in zip(axes, compare_clouds.items()):
    for ax, (view_name, extrinsic) in zip(row, VIEWS.items()):
        ax.imshow(render([cloud, camera_frame], extrinsic))
        ax.set_title(f"{name} - {view_name}")
        ax.axis("off")
fig.savefig(os.path.join(FIG_DIR, f"local_clouds_pair_{COMPARE_ID}.png"), dpi=150)
plt.show()

# descomentar para ver la nube
#for name, cloud in compare_clouds.items():
#    o3d.visualization.draw_geometries([cloud, camera_frame], window_name=f"par {COMPARE_ID} - {name}")

In [ ]:
# se ve que CREStereo es por bastante el mejor al generar la nube local, vamos con ese
CLOUD_METHOD = "crestereo"

local_clouds = {i: local_point_cloud(disparities[CLOUD_METHOD][i], rectified[i][0]) for i in capture_ids}
for i, cloud in local_clouds.items():
    print(f"par {i}: {len(cloud.points)} puntos")

### 4.2.3 Pose de la camara

In [ ]:
BOARD = cv2.aruco.CharucoBoard((9, 13), 42.0, 31.5, cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_5X5_100))
MIN_CHARUCO_CORNERS = 6

# detectamos sobre la izquierda rectificada, mismo sistema que las nubes locales
K_rect = P1[:3, :3]
dist_rect = np.zeros(5)

charuco_params = cv2.aruco.CharucoParameters()
charuco_params.cameraMatrix = K_rect
charuco_params.distCoeffs = dist_rect
charuco_params.minMarkers = 1
charuco_detector = cv2.aruco.CharucoDetector(BOARD, charuco_params)

# pares (punto 3D en el tablero, pixel en la imagen)
def board_correspondences(gray):
    charuco_corners, charuco_ids, marker_corners, marker_ids = charuco_detector.detectBoard(gray)
    
    # caso general, abajo esta el handler de pocas esquinas detecadas
    if charuco_ids is not None and len(charuco_ids) >= MIN_CHARUCO_CORNERS:
        obj_points, img_points = BOARD.matchImagePoints(charuco_corners, charuco_ids)
        return obj_points.reshape(-1, 3), img_points.reshape(-1, 2), "charuco"
    # si no hay suficientes esquinas del damero, usamos las 4 esquinas de cada marcador
    obj_points = np.concatenate([BOARD.getObjPoints()[k] for k in marker_ids.ravel()])
    img_points = np.concatenate(marker_corners).reshape(-1, 2)
    return obj_points, img_points, "marcadores"

# pose del tablero respecto de la camara
def camera_pose(rect_l):
    obj_points, img_points, source = board_correspondences(cv2.cvtColor(rect_l, cv2.COLOR_BGR2GRAY))
    _, rvec, tvec = cv2.solvePnP(obj_points, img_points, K_rect, dist_rect) # calculo la pose de la camara
    proj = cv2.projectPoints(obj_points, rvec, tvec, K_rect, dist_rect)[0].reshape(-1, 2) # proyecto para checkear errores
    rms = np.sqrt(np.mean(np.sum((proj - img_points) ** 2, axis=1))) # calculo rmse
    return cv2.Rodrigues(rvec)[0], tvec.ravel(), rms, len(obj_points), source

In [ ]:
board_center = np.array([*BOARD.getChessboardSize(), 0]) * BOARD.getSquareLength() / 2

# detector con los parametros por defecto para comparar (sin K y minMarkers = 2)
default_detector = cv2.aruco.CharucoDetector(BOARD)
def count_corners(detector, rect):
    ids = detector.detectBoard(cv2.cvtColor(rect, cv2.COLOR_BGR2GRAY))[1]
    return 0 if ids is None else len(ids)

poses = {}
print(f"{'par':>3} {'esq. defecto':>13} {'esq. ajustado':>14} {'fuente':>11} {'puntos':>7} {'RMS [px]':>9} {'dist [mm]':>10}")
for i in capture_ids:
    R_cb, t_cb, rms, n_points, source = camera_pose(rectified[i][0])
    poses[i] = (R_cb, t_cb)
    distance = np.linalg.norm(R_cb @ board_center + t_cb)
    n_default, n_tuned = count_corners(default_detector, rectified[i][0]), count_corners(charuco_detector, rectified[i][0])
    print(f"{i:>3} {n_default:>13} {n_tuned:>14} {source:>11} {n_points:>7} {rms:>9.2f} {distance:>10.1f}")

In [ ]:
SHOW_POSE_IDS = [0, 7, 10]

fig, axes = plt.subplots(1, len(SHOW_POSE_IDS), figsize=(18, 3.6), layout="constrained")
for ax, i in zip(axes, SHOW_POSE_IDS):
    R_cb, t_cb = poses[i]
    # ejes dibujados en el centro del tablero (el origen real es la esquina)
    img = cv2.drawFrameAxes(rectified[i][0].copy(), K_rect, dist_rect, cv2.Rodrigues(R_cb)[0], R_cb @ board_center + t_cb, 126, 6)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"par {i}")
    ax.axis("off")
fig.savefig(os.path.join(FIG_DIR, "camera_poses.png"), dpi=150)
plt.show()

### Correccion de disparidad

In [ ]:
# en un par bien rectificado un mismo punto cae en la misma fila de las dos imagenes
calib_dy = np.concatenate([
    cv2.undistortPoints(l.reshape(-1, 1, 2), K_l, dist_l, R=maps["R1"], P=P1)[:, 0, 1]
    - cv2.undistortPoints(r.reshape(-1, 1, 2), K_r, dist_r, R=maps["R2"], P=P2)[:, 0, 1]
    for l, r in zip(corners_l, corners_r)
])

def charuco_rows(rect):
    corners, ids, _, _ = charuco_detector.detectBoard(cv2.cvtColor(rect, cv2.COLOR_BGR2GRAY))
    return {} if ids is None else dict(zip(ids.ravel(), corners.reshape(-1, 2)[:, 1]))

capture_dy = []
for i in capture_ids:
    rows_l, rows_r = charuco_rows(rectified[i][0]), charuco_rows(rectified[i][1])
    capture_dy += [rows_l[k] - rows_r[k] for k in rows_l.keys() & rows_r.keys()]
capture_dy = np.array(capture_dy)

print(f"{'':<12} {'esquinas':>9} {'media dy [px]':>14} {'mediana |dy| [px]':>18}")
for name, dy in {"calibración": calib_dy, "capturas": capture_dy}.items():
    print(f"{name:<12} {len(dy):>9} {dy.mean():>14.2f} {np.median(np.abs(dy)):>18.2f}")

In [ ]:
BASELINE = -P2[0, 3] / P2[0, 0]

# disparidad que predice PnP en las esquinas del tablero y la que midio el metodo
def board_disparities(i, method=CLOUD_METHOD):
    obj_points, img_points, _ = board_correspondences(cv2.cvtColor(rectified[i][0], cv2.COLOR_BGR2GRAY))
    R_cb, t_cb = poses[i]
    z_pnp = (obj_points @ R_cb.T + t_cb)[:, 2]
    cols, rows = np.round(img_points).astype(int).T
    return K_rect[0, 0] * BASELINE / z_pnp, disparities[method][i][rows, cols]

def disparity_offset(i):
    expected, measured = board_disparities(i)
    return np.median(expected - measured)

# esquinas del tablero trianguladas con la disparidad (sistema de la camara)
def stereo_board_points(i, disparity):
    obj_points, img_points, _ = board_correspondences(cv2.cvtColor(rectified[i][0], cv2.COLOR_BGR2GRAY))
    cols, rows = np.round(img_points).astype(int).T
    points = cv2.perspectiveTransform(np.c_[img_points, disparity[rows, cols]].reshape(-1, 1, 3), Q).reshape(-1, 3)
    return obj_points, points

# distancias entre esquinas trianguladas / distancias reales
def board_scale(i, disparity):
    obj_points, points = stereo_board_points(i, disparity)
    real = np.linalg.norm(obj_points[:, None] - obj_points[None], axis=2)
    stereo = np.linalg.norm(points[:, None] - points[None], axis=2)
    far = real > 100
    return np.median(stereo[far] / real[far])

# altura de las esquinas trianguladas en el sistema del tablero, deberia dar 0
def board_height(i, disparity):
    _, points = stereo_board_points(i, disparity)
    R_cb, t_cb = poses[i]
    return np.median(((points - t_cb) @ R_cb)[:, 2])

# calculamos cuanto es el offset de disparidad (error de la camara entre dias de fotos) y sumamos a las nubes
disparity_offsets = {i: disparity_offset(i) for i in capture_ids}
print(f"{'par':>3} {'offset [px]':>12} {'escala antes':>13} {'escala después':>15} {'Z tablero antes [mm]':>21} {'Z tablero después [mm]':>23}")
for i in capture_ids:
    disparity = disparities[CLOUD_METHOD][i]
    corrected = disparity + disparity_offsets[i]
    print(f"{i:>3} {disparity_offsets[i]:>12.2f} {board_scale(i, disparity):>13.4f} {board_scale(i, corrected):>15.4f} "
          f"{board_height(i, disparity):>21.1f} {board_height(i, corrected):>23.1f}")

local_clouds = {i: local_point_cloud(disparities[CLOUD_METHOD][i] + disparity_offsets[i], rectified[i][0]) for i in capture_ids}

In [ ]:
# si fuera un error de escala el residuo creceria con la disparidad, si es un corrimiento es constante
# los residuos muy negativos son esquinas tapadas por el auto (se mide la disparidad del auto, mas cercano)
Y_RANGE = (-15, 25)
colors = plt.colormaps["tab20"](np.linspace(0, 1, len(capture_ids)))
fig, ax = plt.subplots(figsize=(8, 4.2), layout="constrained")
hidden = 0
for i, color in zip(capture_ids, colors):
    expected, measured = board_disparities(i)
    ax.scatter(expected, expected - measured, s=10, color=color)
    hidden += np.sum(expected - measured < Y_RANGE[0])
ax.set_ylim(*Y_RANGE)
ax.text(0.99, 0.03, f"{hidden} esquinas tapadas por el auto quedan debajo del gráfico", transform=ax.transAxes, ha="right", fontsize=9)
d = np.array([60, 440])
ax.plot(d, 0.05 * d, "k--", lw=1, label="error de escala del 5 %")
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("disparidad predicha por PnP [px]")
ax.set_ylabel("predicha − medida [px]")
ax.set_title("Esquinas del tablero en las 15 vistas (un color por vista)")
ax.legend()
fig.savefig(os.path.join(FIG_DIR, "disparity_offset.png"), dpi=150)
plt.show()

In [ ]:
# cuantas esquinas del tablero tienen disparidad y cuanto se dispersa el residuo sin el offset de cada vista
print(f"{'método':<10} {'esquinas con disparidad':>24} {'corrimiento [px]':>17} {'dispersión [px]':>16}")
for name in METHODS:
    valid, total, residuals, spread = 0, 0, [], []
    for i in capture_ids:
        expected, measured = board_disparities(i, name)
        ok = measured > 0
        residual = expected[ok] - measured[ok]
        residuals.append(residual)
        spread.append(residual - np.median(residual))
        valid, total = valid + ok.sum(), total + len(ok)
    spread = np.concatenate(spread)
    print(f"{name:<10} {f'{valid}/{total}':>24} {np.median(np.concatenate(residuals)):>17.2f} {1.4826 * np.median(np.abs(spread)):>16.2f}")

### 4.2.4 Transformacion al sistema del tablero

In [ ]:
# inversa de la pose, X_board = R_cb^T (X_cam - t_cb)
def camera_to_world(R_cb, t_cb):
    T_wc = np.eye(4)
    T_wc[:3, :3] = R_cb.T
    T_wc[:3, 3] = -R_cb.T @ t_cb
    return T_wc

# piramide de alambre en la posicion de la camara (Open3D recibe la pose tablero -> camara)
def camera_frustum(R_cb, t_cb, scale=40):
    extrinsic = np.eye(4)
    extrinsic[:3, :3] = R_cb
    extrinsic[:3, 3] = t_cb
    return o3d.geometry.LineSet.create_camera_visualization(1920, 1080, K_rect, extrinsic, scale)

world_clouds = {i: o3d.geometry.PointCloud(local_clouds[i]).transform(camera_to_world(*poses[i])) for i in capture_ids}
frustums = [camera_frustum(*poses[i]) for i in capture_ids]
board_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=100)

In [ ]:
RENDER_EVERY = 10
WORLD_VIEWS = {"arriba": view_extrinsic(board_center, 0, 0, 900), "oblicua": view_extrinsic(board_center, 0, 50, 900)}

scene = [world_clouds[i].uniform_down_sample(RENDER_EVERY) for i in capture_ids] + frustums + [board_frame]
fig, axes = plt.subplots(1, len(WORLD_VIEWS), figsize=(16, 4.6), layout="constrained")
for ax, (view_name, extrinsic) in zip(axes, WORLD_VIEWS.items()):
    ax.imshow(render(scene, extrinsic))
    ax.set_title(view_name)
    ax.axis("off")
fig.savefig(os.path.join(FIG_DIR, "world_scene.png"), dpi=150)
plt.show()

In [ ]:
# cada vista pintada de un color, si encajan los colores se mezclan sobre un mismo auto
CHECK_COLORS = {0: [1, 0, 0], 7: [0, 0.7, 0], 10: [0, 0, 1]}

# solo para la figura, nos quedamos con lo que esta sobre el tablero sin el piso (15 mm de margen)
board_w, board_h = np.array(BOARD.getChessboardSize()) * BOARD.getSquareLength()
board_box = o3d.geometry.AxisAlignedBoundingBox([0, 0, -150], [board_w, board_h, -15])
painted = [world_clouds[i].crop(board_box).paint_uniform_color(color) for i, color in CHECK_COLORS.items()]
car_center = np.median(np.concatenate([np.asarray(c.points) for c in painted]), axis=0)
CHECK_VIEWS = {"arriba": view_extrinsic(car_center, 0, 0, 400), "lateral": view_extrinsic(car_center, 90, -90, 400)}

fig, axes = plt.subplots(1, len(CHECK_VIEWS), figsize=(16, 4.6), layout="constrained")
for ax, (view_name, extrinsic) in zip(axes, CHECK_VIEWS.items()):
    ax.imshow(render(painted + [board_frame], extrinsic))
    ax.set_title(f"vistas {list(CHECK_COLORS)} - {view_name}")
    ax.axis("off")
fig.savefig(os.path.join(FIG_DIR, "world_alignment.png"), dpi=150)
plt.show()

## 4.3 Fusion y optimizacion

### Fusion y recorte

In [ ]:
FLOOR_MARGIN = 5.0
CAR_MAX_HEIGHT = 150.0

def crop(cloud, box):
    return cloud.select_by_index(box.get_point_indices_within_bounding_box(cloud.points))

# caja alineada con el tablero, desde 5 mm sobre el piso hasta 15 cm de alto (Z negativo es hacia arriba)
car_box = o3d.geometry.OrientedBoundingBox(
    [board_w / 2, board_h / 2, -(CAR_MAX_HEIGHT + FLOOR_MARGIN) / 2], np.eye(3), [board_w, board_h, CAR_MAX_HEIGHT - FLOOR_MARGIN]
)

fused = o3d.geometry.PointCloud()
for i in capture_ids:
    fused += crop(world_clouds[i], car_box)
print("puntos fusionados:", len(fused.points))

### Efecto del filtro de bordes

In [ ]:
EDGE_ID = 7

disparity = disparities[CLOUD_METHOD][EDGE_ID] + disparity_offsets[EDGE_ID]
edge_clouds = {
    "sin filtro": crop(local_point_cloud(disparity, rectified[EDGE_ID][0], np.inf).transform(camera_to_world(*poses[EDGE_ID])), car_box),
    "con filtro": crop(local_point_cloud(disparity, rectified[EDGE_ID][0]).transform(camera_to_world(*poses[EDGE_ID])), car_box),
}
edge_center = np.median(np.asarray(edge_clouds["con filtro"].points), axis=0)
EDGE_VIEWS = {"lateral": view_extrinsic(edge_center, 90, -90, 230), "oblicua": view_extrinsic(edge_center, 40, -50, 230)}

fig, axes = plt.subplots(len(edge_clouds), len(EDGE_VIEWS), figsize=(12, 3.4 * len(edge_clouds)), layout="constrained")
for row, (name, cloud) in zip(axes, edge_clouds.items()):
    for ax, (view_name, extrinsic) in zip(row, EDGE_VIEWS.items()):
        ax.imshow(render([cloud], extrinsic))
        ax.set_title(f"par {EDGE_ID} {name} ({len(cloud.points)} pts) - {view_name}")
        ax.axis("off")
fig.savefig(os.path.join(FIG_DIR, "edge_filter.png"), dpi=150)
plt.show()

### Submuestreo y outliers

In [ ]:
VOXEL_SIZE = 1.0

downsampled = fused.voxel_down_sample(VOXEL_SIZE)
filtered, _ = downsampled.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)

# lo que queda suelto forma grupos separados del auto, nos quedamos con el grupo mas grande
def largest_cluster(cloud):
    labels = np.array(cloud.cluster_dbscan(eps=3.0, min_points=10))
    return cloud.select_by_index(np.where(labels == np.bincount(labels[labels >= 0]).argmax())[0])

car = largest_cluster(filtered)

for name, cloud in {"fusionada": fused, "voxel": downsampled, "outliers": filtered, "cluster": car}.items():
    print(f"{name:<10} {len(cloud.points):>10} puntos")

In [ ]:
car_center = np.median(np.asarray(car.points), axis=0)
CAR_VIEWS = {"arriba": view_extrinsic(car_center, 0, 0, 320), "lateral": view_extrinsic(car_center, 90, -90, 320),
             "oblicua": view_extrinsic(car_center, 40, -50, 320)}
# la fusionada se muestra con 1 de cada 10 puntos, el resto completas
STAGES = {"recorte": fused, "voxel": downsampled, "outliers": filtered, "dbscan": car}

# una columna por etapa y una fila por vista
fig, axes = plt.subplots(len(CAR_VIEWS), len(STAGES), figsize=(18, 2.9 * len(CAR_VIEWS)), layout="constrained")
for col, (stage, cloud) in enumerate(STAGES.items()):
    shown = cloud.uniform_down_sample(RENDER_EVERY) if cloud is fused else cloud
    for row, (view_name, extrinsic) in enumerate(CAR_VIEWS.items()):
        axes[row, col].imshow(render([shown], extrinsic))
        axes[row, col].set_title(f"{stage} ({len(cloud.points)} pts) - {view_name}")
        axes[row, col].axis("off")
fig.savefig(os.path.join(FIG_DIR, "fusion_cleanup.png"), dpi=150)
plt.show()

### ICP (probado y descartado)

In [ ]:
ICP_MAX_DIST = 3.0

# cada vista contra la union de las otras 14, si las poses fueran malas ICP la moveria y bajaria la distancia
car_views = {i: crop(world_clouds[i], car_box).voxel_down_sample(VOXEL_SIZE) for i in capture_ids}
print(f"{'par':>3} {'dist. antes [mm]':>17} {'dist. después [mm]':>19} {'giro ICP [°]':>13}")
for i in capture_ids:
    others = o3d.geometry.PointCloud()
    for j in capture_ids:
        if j != i:
            others += car_views[j]
    others = others.voxel_down_sample(VOXEL_SIZE)
    view = o3d.geometry.PointCloud(car_views[i])
    before = np.median(np.asarray(view.compute_point_cloud_distance(others)))
    icp = o3d.pipelines.registration.registration_icp(view, others, ICP_MAX_DIST)
    after = np.median(np.asarray(view.transform(icp.transformation).compute_point_cloud_distance(others)))
    angle = np.degrees(np.linalg.norm(cv2.Rodrigues(icp.transformation[:3, :3])[0]))
    print(f"{i:>3} {before:>17.2f} {after:>19.2f} {angle:>13.2f}")

### Exportacion

In [ ]:
PLY_PATH = "ferrari.ply"
o3d.io.write_point_cloud(PLY_PATH, car, write_ascii=False)
print(f"{PLY_PATH}: {len(car.points)} puntos, {os.path.getsize(PLY_PATH) / 1e6:.1f} MB")

## 5 Medicion

In [ ]:
TRIM = 0.1 # % de puntos que se ignoran en cada extremo

# ejes del auto sobre el piso (PCA en XY), el piso es Z = 0 y la altura es hacia Z negativo
def measure(points, trim=TRIM):
    xy_mean = points[:, :2].mean(axis=0)
    _, _, car_axes = np.linalg.svd(points[:, :2] - xy_mean, full_matrices=False)
    uv = (points[:, :2] - xy_mean) @ car_axes.T
    low, high = np.percentile(uv, [trim, 100 - trim], axis=0)
    height = -np.percentile(points[:, 2], trim)
    return xy_mean, car_axes, low, high, height

xy_mean, car_axes, low, high, height = measure(np.asarray(car.points))
length, width = high - low

# medidas oficiales del LEGO 42212 (redondeadas al cm)
REAL_SIZE = {"largo": 290.0, "ancho": 120.0, "alto": 80.0}

for name, measured in zip(REAL_SIZE, [length, width, height]):
    error = 100 * (measured - REAL_SIZE[name]) / REAL_SIZE[name]
    print(f"{name:<6} medido {measured:6.1f} mm   real {REAL_SIZE[name]:6.1f} mm   error {error:+5.1f} %")

In [ ]:
R_car = np.eye(3)
R_car[:2, :2] = car_axes.T
R_car[:, 2] = np.cross(R_car[:, 0], R_car[:, 1])
box_center = [*(xy_mean + ((low + high) / 2) @ car_axes), -height / 2]
size_box = o3d.geometry.OrientedBoundingBox(box_center, R_car, [length, width, height])
box_lines = o3d.geometry.LineSet.create_from_oriented_bounding_box(size_box).paint_uniform_color([0, 0, 0])

fig, axes = plt.subplots(1, len(CAR_VIEWS), figsize=(18, 3.4), layout="constrained")
for ax, (view_name, extrinsic) in zip(axes, CAR_VIEWS.items()):
    ax.imshow(render([car, box_lines], extrinsic))
    ax.set_title(view_name)
    ax.axis("off")
fig.savefig(os.path.join(FIG_DIR, "measurement.png"), dpi=150)
plt.show()

## Ablacion

In [ ]:
# rearma el auto con algun paso apagado para ver cuanto aporta cada uno a la medicion
def build_car(offsets, max_gradient=MAX_DISP_GRADIENT, use_dbscan=True):
    fused = o3d.geometry.PointCloud()
    for i in capture_ids:
        local = local_point_cloud(disparities[CLOUD_METHOD][i] + offsets[i], rectified[i][0], max_gradient)
        fused += crop(local.transform(camera_to_world(*poses[i])), car_box)
    cloud, _ = fused.voxel_down_sample(VOXEL_SIZE).remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
    return largest_cluster(cloud) if use_dbscan else cloud

def size_row(name, points, trim=TRIM):
    _, _, low, high, height = measure(points, trim)
    sizes = [*(high - low), height]
    errors = [100 * (s - r) / r for s, r in zip(sizes, REAL_SIZE.values())]
    print(f"{name:<22} {len(points):>8} " + " ".join(f"{s:6.1f} ({e:+5.1f}%)" for s, e in zip(sizes, errors)))

no_offsets = {i: 0.0 for i in capture_ids}
variants = {
    "completo": car,
    "sin corrección": build_car(no_offsets),
    "sin filtro de bordes": build_car(disparity_offsets, np.inf),
    "sin DBSCAN": build_car(disparity_offsets, use_dbscan=False),
    "gradiente < 0.5": build_car(disparity_offsets, 0.5),
    "gradiente < 2": build_car(disparity_offsets, 2.0),
}

print(f"{'variante':<22} {'puntos':>8} {'largo [mm]':>15} {'ancho [mm]':>15} {'alto [mm]':>15}")
for name, cloud in variants.items():
    size_row(name, np.asarray(cloud.points))
print()
for trim in [0, 0.1, 0.5, 1.0]:
    size_row(f"recorte {trim} %", np.asarray(car.points), trim)

In [ ]:
# auto sin y con la correccion de disparidad, contra una caja con las medidas oficiales
real_box = o3d.geometry.OrientedBoundingBox(box_center, R_car, list(REAL_SIZE.values()))
real_lines = o3d.geometry.LineSet.create_from_oriented_bounding_box(real_box).paint_uniform_color([0, 0, 0])
compare = {"sin corrección": variants["sin corrección"], "con corrección": car}

# recorta el blanco que sobra, igual en las dos imagenes para que queden a la misma escala
def crop_white(images):
    rows, cols = np.nonzero(np.any(np.stack(images).min(axis=3) < 0.99, axis=0))
    return [image[rows.min() - 5:rows.max() + 5, cols.min() - 5:cols.max() + 5] for image in images]

fig, axes = plt.subplots(2, 2, figsize=(12, 4.6), layout="constrained")
for col, view_name in enumerate(["arriba", "lateral"]):
    images = crop_white([render([cloud, real_lines], CAR_VIEWS[view_name]) for cloud in compare.values()])
    for ax, name, image in zip(axes[:, col], compare, images):
        ax.imshow(np.rot90(image) if view_name == "arriba" else image)
        ax.set_title(f"{name} - {view_name}")
        ax.axis("off")
fig.savefig(os.path.join(FIG_DIR, "disparity_correction.png"), dpi=150)
plt.show()